<a href="https://colab.research.google.com/github/Swodhin/genai-lfconnect/blob/main/week-1/class-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 3 — Calling APIs & Your First AI Script
**Week 1: Python Foundations for Generative AI — "The Interpreter"**

### Learning objectives
By the end of this notebook you will be able to:
- Explain what an API call is in terms of client, server, request, and response
- Read secrets safely from environment variables
- Make an HTTP request with the `requests` library
- Describe the anatomy of an LLM chat completion request
- Write a complete script that sends a prompt to a real LLM API and prints the response

**This notebook needs an API key to run the live demo cells — running in Google Colab:**
1. Get a free key from https://console.groq.com/keys
2. In Colab, click the key icon (🔑 "Secrets") in the left sidebar
3. Add a new secret named `GROQ_API_KEY`, paste your key as the value, and toggle **Notebook access** on
4. Run the setup cell below — it reads the secret automatically

Never paste a real key directly into a notebook cell or commit one to git.

In [1]:
!pip install -q groq requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.9 MB/s eta 0:00:00


## Setup
We read the key from Colab's Secrets manager (falling back to a plain environment variable if you're running this outside Colab) — never hardcoded — and print a friendly message if it's missing so the rest of the notebook can still be read (and partially run) without one.

In [3]:
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    print(
        "No API key found.\n"
        "In Colab: add a secret named GROQ_API_KEY via the 🔑 Secrets panel (left sidebar) and enable notebook access.\n"
        "Elsewhere: set GROQ_API_KEY as an environment variable before launching Jupyter.\n"
        "You can still read and run the non-API cells in this notebook."
    )
else:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY  # so later cells can use os.environ.get consistently
    print("Found GROQ_API_KEY.")

Found GROQ_API_KEY.


## 1. What Is an API? Client, Server, Request, Response
An API is a contract: send a request in an expected shape, get a response back. HTTP is the protocol underneath — your script is the *client*, the AI provider runs the *server*. Every request has a **method** (GET/POST), a **URL**, **headers**, and often a **body**. Every response has a **status code** and a body. Calling an LLM API is just: POST a JSON body to an endpoint, get a JSON body back.

In [ ]:
# A quick mental model — no network call yet, just the shape of things:
request_shape = {
    "method": "POST",
    "url": "https://api.groq.com/openai/v1/chat/completions",
    "headers": {"Authorization": "Bearer <API_KEY>", "Content-Type": "application/json"},
    "body": {"model": "llama-3.3-70b-versatile", "messages": [{"role": "user", "content": "Hi"}]},
}
import json
print(json.dumps(request_shape, indent=2))

## 2. Secrets & Environment Variables
API keys are secrets — anyone with your key can spend your money and act as you. Never hardcode a key in a script, and never commit one to git. `os.environ.get(...)` returns `None` (not an error) if a variable isn't set, so always check before using it.

In [ ]:
def safe_get_env(name):
    value = os.environ.get(name)
    if not value:
        print(f"Missing {name} — set it as an environment variable before continuing.")
    return value

_ = safe_get_env("GROQ_API_KEY")

## 3. The `requests` Library Basics
`requests` is Python's standard tool for making HTTP calls (`pip install requests`). `requests.get(url)` fetches data; `requests.post(url, json=payload)` sends a JSON body. `response.status_code` tells you if it worked; `response.json()` parses the JSON reply. Always set a `timeout=`.

In [ ]:
import requests

# httpbin.org is a free public test API — safe to call without any key.
resp = requests.get("https://httpbin.org/get", params={"course": "genai-foundations"}, timeout=10)
resp2 = requests.get("https://httpbin.org/get", params={"course": "genai-foundations"}, timeout=10)

print("status:", resp.status_code)
print(resp.json()["args"])

## 4. Anatomy of a Chat Completion Request
The body of an LLM API call carries the model name and a list of role-tagged messages (`"system"`, `"user"`, `"assistant"`), plus settings like `temperature`. The reply nests the model's text inside the response body; an SDK unwraps it for you.

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise Python tutor."},
    {"role": "user", "content": "What is a variable, in one sentence?"},
]
print(json.dumps(messages, indent=2))

## 5. Reading the Response, Safely
Wrap network calls in `try/except` — they fail in ways your code can't prevent. Check status before trusting the data, and set a `max_tokens` limit so a runaway response doesn't blow your budget.

In [4]:
def call_llm(prompt, model="llama-3.3-70b-versatile", max_tokens=200):
    """Send a single-turn prompt to Groq and return the text, or an error message."""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return "Error: GROQ_API_KEY is not set."
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            tempertarue = 5,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Request failed: {e}"

## 6. Demo: Your First AI Script, End to End
This cell needs a real `GROQ_API_KEY` in your environment to actually call the model. Groq's API is OpenAI-compatible, so if you'd rather use OpenAI's own SDK, see the commented alternative below the main script — the shape of the call barely changes.

In [5]:
if os.environ.get("GROQ_API_KEY"):
    reply = call_llm("Explain what a variable is, in one sentence, for a total beginner.")
    print(reply)
else:
    print("Set GROQ_API_KEY in your environment, then re-run this cell to see a live response.")

A variable is a container that stores a value, like a name or a number, so you can use and manipulate it in a program, and you can think of it like a labeled box where you can store and retrieve information.


In [ ]:
# --- OpenAI alternative (Groq's API is OpenAI-compatible, so the SDK swap is small) ---
# import os
# from openai import OpenAI
#
# api_key = os.environ.get("OPENAI_API_KEY")
# client = OpenAI(api_key=api_key)
# response = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": "Explain what a variable is, in one sentence."}],
# )
# print(response.choices[0].message.content)

### Week 1, closed
You've now written a complete, working AI script — from `x = 5` to a live model response. That's Week 1: you read Python fluently enough to keep Week 2's ideas about how LLMs actually work grounded in real, runnable code.

## Challenges
Work through these in order. No solutions are provided — each starter cell has a `# TODO` marking where your code goes. Challenges 2 and 4 need network access; challenge 4's live call needs your own API key.

### Challenge 1 — `safe_get_env` Helper
Write a function `safe_get_env(name, default=None)` that reads an environment variable and prints a friendly, specific message if it's missing (including the variable name), returning `default` in that case. Test it on a variable name that does *not* exist.

**Acceptance criteria:** calling it on a missing variable prints a helpful message and returns the default without raising an exception.

In [ ]:
# TODO: write safe_get_env(name, default=None) and test it on a nonexistent variable name


### Challenge 2 — GET Request to a Public Test API
Make a GET request to `https://httpbin.org/uuid` (no key required) and print the status code and the parsed JSON response.

**Acceptance criteria:** prints a status code of 200 and a dict containing a `"uuid"` key.

In [ ]:
# TODO: GET https://httpbin.org/uuid and print status code + JSON


### Challenge 3 — Build a 3-Turn Message List
Construct a Python list of message dicts representing a 3-turn conversation: a system message, a user message, an assistant message, and a final new user message (4 messages total). Print it as formatted JSON.

**Acceptance criteria:** the list has exactly 4 dicts, each with `"role"` and `"content"` keys, roles alternating sensibly.

In [ ]:
# TODO: build the 4-message conversation list and print it as JSON


### Challenge 4 — `call_llm(prompt)` Wrapper Function
Modify the `call_llm` function from section 5 (copy it into the cell below) so it also accepts a `system_prompt` parameter and includes it as the first message when provided. Test it with your own API key.

**Acceptance criteria:** the function sends a system message when `system_prompt` is given, and still works (returns model text) when it is omitted.

In [ ]:
# TODO: copy call_llm here and add a system_prompt parameter, then test it


### Challenge 5 (Stretch) — Prompt-Response Logger
Write a script that: reads a prompt via `input()`, calls `call_llm(prompt)`, and appends a dict `{"prompt": ..., "response": ...}` to a JSON file called `chat_log.json` (creating it if it doesn't exist, appending to the existing list otherwise).

**Acceptance criteria:** running the cell twice with different prompts results in `chat_log.json` containing a list with 2 entries, each with `prompt` and `response` keys.

In [ ]:
# TODO: read a prompt, call the LLM, and append the exchange to chat_log.json
